In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time
import os


# -------------------------------------------------------------
# setup browser
# -------------------------------------------------------------
def init_browser():
    print("Initialize the browser...")

    options = webdriver.ChromeOptions()
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--start-maximized')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)

    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    return driver



#To find the list,tried 5 kinds of selector.Class_name do work.

In [2]:
def find_list_items(driver, saved_selector):
    print("Find earthquake list items...")

    # if we have a saved selector, try it first
    if saved_selector["selector"] and saved_selector["type"]:
        try:
            items = driver.find_elements(saved_selector["type"], saved_selector["selector"])
            if items:
                valid = [i for i in items if i.text.strip()]
                if valid:
                    print(f"Use the saved selector to find { Len (valid)} list items")
                    return valid
        except:
            print("The saved selector is invalid. Try again...")

    # multiple strategies to find list items
    list_item_selectors = [
        (By.CLASS_NAME, "DesignatedCatalogue_list-item__axM47"),
        # (By.CSS_SELECTOR, "[class*='list-item']"),
        # (By.XPATH, "//div[contains(@class, 'list-item')]"),
        # (By.XPATH, "//div[contains(text(), 'M') or contains(text(), '级')]"),
        # (By.XPATH, "//div[contains(@class, 'DesignatedCatalogue')]"),
    ]

    for selector_type, selector_value in list_item_selectors:
        try:
            items = driver.find_elements(selector_type, selector_value)
            valid_items = [i for i in items if i.text.strip()]
            if valid_items:
                #print(f"选择器 '{selector_value}' 找到 {len(valid_items)} 项")
                saved_selector["selector"] = selector_value
                saved_selector["type"] = selector_type
                return valid_items
        except:
            continue

    
    return []

#To extract features,after click the current list.Finding the time,Longitude and Latitude，Hypocentral Depth，Earthquake Magnitude
#the second time add the feature:title

In [3]:
def extract_features(driver, count, item_text):
    xpath = '//*[@id="root"]/div/div/div/div[2]/div/div[2]/div/div/div/div[1]/div[2]'

    try:
        container = driver.find_element(By.XPATH, xpath)
        lines = [l.strip() for l in container.text.split("\n") if l.strip()]

        if len(lines) >= 4:
            return {
                'order': count + 1,
                'text': item_text,    # ←★ save the original item text
                'time': lines[0],
                'Latitude and longitude': lines[1],
                'Depth of focus': lines[2],
                'Series': lines[3]
            }
        else:
            print("    Less than 4 lines")
            return None

    except Exception as e:
        print(f"    Feature extraction failure: { e } ")
        return None


#Whether it is affected by network speed or not, the sleep time must be more than 5 seconds; otherwise, the content will fail to load, leading to the failure to locate the element.

In [4]:
def wait_for_list_page(driver, saved_selector):
    print("waiting for the page...")
    time.sleep(7)

    for _ in range(5):
        items = find_list_items(driver, saved_selector)
        if items:
            print("the list already load")
            return True
        time.sleep(1)

    print("fail")
    return False

#put 'item_text' into features

In [5]:
def click_and_extract(driver, item, index, data_list):
    print(f"\n is dealing with the { index + 1} th earthquake...")

    try:
        # added to pass item_text to extract_features
        item_text = item.text.strip()
        print(f"  text: {item_text}")

        print("  click list...")
        item.click()
        time.sleep(3)

        print("  Extract details page features...")

        # ★ inject item_text into extract_features
        features = extract_features(driver, len(data_list), item_text)

        if features:
            data_list.append(features)
            print(f" ✓ successfully extracted { features [ 'time' ]} | { features [ 'series' ]} level")
            return True
        else:
            print("  Extraction failed")
            return False

    except Exception as e:
        print(f"  Processing failure: { e } ")
        return False


#Do not reload the list before crawling reaches the last max_items. Reloading takes a long time and is prone to errors. Pause for 5 seconds to prevent anti-crawling measures.
#The first crawl was conducted on December 9, with a total count of 1,344. The second crawl took place on December 12, and the count increased to 1,356.


In [6]:
def crawl_earthquakes(driver, max_items=1356):
    saved_selector = {"selector": None, "type": None}
    data_list = []

    list_url = "https://www.cenc.ac.cn/earthquake-manage-publish-web/designated-catalogue"
    print(f"\n visit list: {list_url}")
    driver.get(list_url)

    if not wait_for_list_page(driver, saved_selector):
        return data_list

    items = find_list_items(driver, saved_selector)
    items_to_crawl = min(len(items), max_items)

    for i in range(items_to_crawl):
        items_now = find_list_items(driver, saved_selector)
        if i >= len(items_now):
            break

        click_and_extract(driver, items_now[i], i, data_list)

        # if i < items_to_crawl - 1:
        #     driver.back()
        #     wait_for_list_page(driver, saved_selector)

        time.sleep(5)

    #print(f"\n共获取 {len(data_list)} 条记录")
    return data_list

In [7]:
def save_to_excel(data):
    if not data:
        print("no data")
        return

    desktop = os.path.join(os.path.expanduser('~'), 'Desktop')
    filename = os.path.join(
        desktop,
        f"the detail data_{time.strftime('%Y%m%d_%H%M%S')}.xlsx"
    )

    df = pd.DataFrame(data)

    # added to reorder columns
    column_order = ['order', 'text', 'time', 'Latitude and longitude', 'Depth of focus', 'Series']
    df = df[[col for col in column_order if col in df.columns]]

    df.to_excel(filename, index=False)
    print(f"save：{filename}")



In [8]:
def main():
    print("="*60)
    print("Data acquisition of China Seismic Network")
    print("="*60)

    driver = init_browser()

    try:
        max_items = input("Number of crawls (default 5) : ").strip()
        max_items = int(max_items) if max_items.isdigit() else 5

        data = crawl_earthquakes(driver, max_items)

        if data:
            save = input("Is the Excel  kept? (y/n default Y) : ").strip().lower()
            if save != "n":
                save_to_excel(data)

    finally:
        driver.quit()
        print("The browser is closed")

In [9]:
if __name__ == "__main__":
    main()

Data acquisition of China Seismic Network
Initialize the browser...

 visit list: https://www.cenc.ac.cn/earthquake-manage-publish-web/designated-catalogue
waiting for the page...
Find earthquake list items...
the list already load
Find earthquake list items...
The saved selector is invalid. Try again...
Find earthquake list items...
The saved selector is invalid. Try again...

 is dealing with the 1 th earthquake...
  text: 2025-12-15 00:30:18四川甘孜州泸定县3.1级地震
  click list...
  Extract details page features...
  Processing failure: 'series' 
Find earthquake list items...
The saved selector is invalid. Try again...

 is dealing with the 2 th earthquake...
  text: 2025-12-13 17:20:00山西忻州市忻府区3.4级地震
  click list...
  Extract details page features...
  Processing failure: 'series' 
Find earthquake list items...
The saved selector is invalid. Try again...

 is dealing with the 3 th earthquake...
  text: 2025-12-13 06:52:16四川成都市彭州市3.9级地震
  click list...
  Extract details page features...
  Proc